<div align="center">
  <h3> 3D Surface Segmentation with MONAI U-Net</h3>
  <img src="https://raw.githubusercontent.com/Mr-Asan/CycleGAN-Pix2pix/main/x.png" width="800"/>
</div>


# 1. Introduction
In this study, a UNet-based segmentation model was trained on 3D surface images using the MONAI (Medical Open Network for AI) library, and the inference process was performed. Below, the Data Analysis, Training, and Inference phases of the study are presented with both conceptual explanations and sample code.

In [ ]:
!pip install /kaggle/input/imagecodes-monai-unet-2/*.whl --no-index



In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from glob import glob
import tifffile as tiff
from tqdm import tqdm

from monai.networks.nets import UNet

import matplotlib.pyplot as plt
import cv2

import imageio
from mpl_toolkits.mplot3d import Axes3D

# 1. Data Analysis
Image and mask data provided in TIFF format were first examined in two dimensions (2D). Image-mask fit and structural features were analyzed using individual slices. Then, to better understand the spatial continuity of the masks, the data were visualized in three dimensions (3D); the volumetric structure of the data was evaluated by overlaying mask surfaces onto translucent image layers.

In [ ]:
def visualize_volume_and_mask(
    image_tif_path,
    mask_tif_path,
    num_slices=25,
    grid_size=5,
    resize_to=128
):
    volume = tiff.imread(image_tif_path)   # (D,H,W)
    mask = tiff.imread(mask_tif_path)       # (D,H,W)

    assert volume.shape == mask.shape, "Image & mask shape mismatch!"

    D, H, W = volume.shape

    slice_indices = np.linspace(0, D - 1, num_slices, dtype=int)

    fig, axes = plt.subplots(
        grid_size, grid_size * 2,
        figsize=(grid_size * 4, grid_size * 2)
    )

    for i, idx in enumerate(slice_indices):
        r = i // grid_size
        c = (i % grid_size) * 2

        # --- IMAGE ---
        img = volume[idx]
        img = cv2.resize(img, (resize_to, resize_to))
        img = (img - img.min()) / (img.max() - img.min() + 1e-6)

        axes[r, c].imshow(img, cmap="gray")
        axes[r, c].set_title(f"Img z={idx}", fontsize=8)
        axes[r, c].axis("off")

        # --- MASK ---
        m = mask[idx]
        m = cv2.resize(
            m,
            (resize_to, resize_to),
            interpolation=cv2.INTER_NEAREST
        )

        axes[r, c + 1].imshow(m, cmap="gray")
        axes[r, c + 1].set_title(f"Mask z={idx}", fontsize=8)
        axes[r, c + 1].axis("off")

    plt.tight_layout()
    plt.show()


In [ ]:
visualize_volume_and_mask(
    image_tif_path="/kaggle/input/vesuvius-challenge-surface-detection/train_images/1004283650.tif",
    mask_tif_path="/kaggle/input/vesuvius-challenge-surface-detection/train_labels/1004283650.tif"
)

In [ ]:
image_path = "/kaggle/input/vesuvius-challenge-surface-detection/train_images/1004283650.tif"
mask_path  = "/kaggle/input/vesuvius-challenge-surface-detection/train_labels/1004283650.tif"

images = tiff.imread(image_path)   
masks  = tiff.imread(mask_path)    

print("Image shape:", images.shape)
print("Mask shape :", masks.shape)


In [ ]:
images = images[:, ::2, ::2]
masks  = masks[:, ::2, ::2]

num_slices = 25
alpha_img = 0.15
alpha_mask = 0.9

D, H, W = images.shape

slice_indices = np.linspace(
    D // 4,
    3 * D // 4 - 1,
    num_slices
).astype(int)

slice_indices = np.clip(slice_indices, 0, D - 1)

view_angles = [0, 90, 180, 270]

In [ ]:
fig = plt.figure(figsize=(25, 6))

for i, azim in enumerate(view_angles):
    ax = fig.add_subplot(1, len(view_angles), i + 1, projection="3d")

    for z in slice_indices:
        Y, X = np.mgrid[0:H, 0:W]
        Z = np.ones_like(X) * z

        img_slice = images[z]
        img_norm = (img_slice - img_slice.min()) / (np.ptp(img_slice) + 1e-6)

        colors_img = plt.cm.gray(img_norm)
        colors_img[..., -1] = alpha_img

        ax.plot_surface(
            X, Y, Z,
            facecolors=colors_img,
            rstride=4,
            cstride=4,
            shade=False
        )
        papyrus_mask = masks[z] == 1

        if papyrus_mask.any():
            ax.scatter(
                X[papyrus_mask],
                Y[papyrus_mask],
                Z[papyrus_mask],
                c="red",
                s=3,
                alpha=alpha_mask
            )

    ax.set_xlim(0, 256)
    ax.set_ylim(0, 256)
    ax.set_zlim(0, D)

    ax.set_xlabel("X")
    ax.set_ylabel("Y")
    ax.set_zlabel("Slice")

    ax.view_init(elev=25, azim=azim)
    ax.set_title(f"Z rotation {azim}°")

plt.tight_layout()
plt.show()

# 3.Training
The MONAI library's 3D UNet architecture was used in the training phase. The model learns the spatial context between successive slices by taking 3D TIFF volumes as input and distinguishes between papyrus and background classes at the voxel level. Cross-entropy loss and the AdamW optimization method were used in the training process.

In [ ]:
class Tiff3DDataset(Dataset):
    def __init__(self, image_paths, mask_paths):
        self.image_paths = image_paths
        self.mask_paths = mask_paths

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img = tiff.imread(self.image_paths[idx])   
        mask = tiff.imread(self.mask_paths[idx])  

        img = img.astype(np.float32)
        img = (img - img.min()) / (img.max() - img.min() + 1e-6)

        img = torch.from_numpy(img).unsqueeze(0)   
        mask = torch.from_numpy(mask).long()       

        return img, mask


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = UNet(
    spatial_dims=3,
    in_channels=1,
    out_channels=2,                # background / papyrus
    channels=(32, 64, 128, 256, 512),
    strides=(2, 2, 2, 2),
    num_res_units=2,
).to(device)


In [ ]:
criterion = nn.CrossEntropyLoss(ignore_index=2)
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-5
)


In [ ]:
train_images = sorted(glob("/kaggle/input/vesuvius-challenge-surface-detection/train_images/*.tif"))
train_masks  = sorted(glob("/kaggle/input/vesuvius-challenge-surface-detection/train_labels/*.tif"))

train_dataset = Tiff3DDataset(train_images, train_masks)

train_loader = DataLoader(
    train_dataset,
    batch_size=1,      
    shuffle=True,
    num_workers=4,
    pin_memory=True
)


In [ ]:
EPOCHS = 150
SAVE_PATH = "unet3d_papyrus.pth"

best_loss = 1e9

for epoch in range(EPOCHS):
    model.train()
    epoch_loss = 0.0

    for imgs, masks in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        imgs = imgs.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    print(f"Epoch {epoch+1} | Loss: {avg_loss:.6f}")

    if avg_loss < best_loss:
        best_loss = avg_loss
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "loss": best_loss
        }, SAVE_PATH)

        print(f"✅ Model saved (loss={best_loss:.6f})")
#This training code runs flawlessly, utilizing 12GB of GPU power. 
#In this section, the training process was performed on a higher-powered device, saving time. The test section presents outputs from the trained model. 


# 3. Inference
The model structure used was visualized, and the testing processes were checked using 2D outputs and 3D spatial visualization of original masks and manufactured synthetic masks.

In [ ]:
checkpoint = torch.load("/kaggle/input/imagecodes-monai-unet-2/unet3d_papyrus.pth", map_location=device)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()


In [ ]:
def predict_3d_volume(model, tif_path):
    vol = tiff.imread(tif_path)
    vol = vol.astype(np.float32)
    vol = (vol - vol.min()) / (vol.max() - vol.min() + 1e-6)

    vol = torch.from_numpy(vol).unsqueeze(0).unsqueeze(0).to(device)

    with torch.no_grad():
        out = model(vol)
        pred = torch.argmax(out, dim=1)

    return pred.squeeze(0).cpu().numpy()


In [ ]:
test_volume = "/kaggle/input/vesuvius-challenge-surface-detection/test_images/1407735.tif"
prediction = predict_3d_volume(model, test_volume)

tiff.imwrite(
    "prediction.tif",
    prediction.astype(np.uint8)
)

print("✅ Prediction saved as prediction.tif")


In [ ]:
import tifffile as tiff
import matplotlib.pyplot as plt
import numpy as np

pred = tiff.imread("prediction.tif")  # (Z, H, W)

num_slices = pred.shape[0]

indices = np.linspace(0, num_slices - 1, 25, dtype=int)

fig, axes = plt.subplots(5, 5, figsize=(10, 10))

for ax, idx in zip(axes.flat, indices):
    ax.imshow(pred[idx], cmap="gray")
    ax.set_title(f"Slice {idx}", fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()
#Predicted Masks

In [ ]:
test_volume = "/kaggle/input/vesuvius-challenge-surface-detection/train_images/102536988.tif"
prediction = predict_3d_volume(model, test_volume)

tiff.imwrite(
    "prediction-2.tif",
    prediction.astype(np.uint8)
)

print("✅ Prediction saved as prediction.tif")


In [ ]:
orig_image_path = "/kaggle/input/vesuvius-challenge-surface-detection/train_images/102536988.tif"
orig_mask_path  = "/kaggle/input/vesuvius-challenge-surface-detection/train_labels/102536988.tif"

pred_image_path = "/kaggle/input/vesuvius-challenge-surface-detection/train_images/102536988.tif"
pred_mask_path  = "prediction-2.tif"

images_orig = tiff.imread(orig_image_path)
masks_orig  = tiff.imread(orig_mask_path)

images_pred = tiff.imread(pred_image_path)
masks_pred  = tiff.imread(pred_mask_path)

images_orig = images_orig[:, ::2, ::2]
masks_orig  = masks_orig[:, ::2, ::2]

images_pred = images_pred[:, ::2, ::2]
masks_pred  = masks_pred[:, ::2, ::2]

num_slices = 25
alpha_img  = 0.15
alpha_mask = 0.9

view_angles = [0, 90, 180, 270]

datasets = [
    ("ORIGINAL", images_orig, masks_orig),
    ("PREDICTED", images_pred, masks_pred),
]

In [ ]:
fig = plt.figure(figsize=(26, 12))

for row, (title_prefix, images, masks) in enumerate(datasets):

    D, H, W = images.shape

    slice_indices = np.linspace(
        D // 4,
        3 * D // 4 - 1,
        num_slices
    ).astype(int)
    slice_indices = np.clip(slice_indices, 0, D - 1)

    for col, azim in enumerate(view_angles):
        ax = fig.add_subplot(
            len(datasets),
            len(view_angles),
            row * len(view_angles) + col + 1,
            projection="3d"
        )

        for z in slice_indices:
            Y, X = np.mgrid[0:H, 0:W]
            Z = np.ones_like(X) * z

            img_slice = images[z]
            img_norm = (img_slice - img_slice.min()) / (np.ptp(img_slice) + 1e-6)

            colors_img = plt.cm.gray(img_norm)
            colors_img[..., -1] = alpha_img

            ax.plot_surface(
                X, Y, Z,
                facecolors=colors_img,
                rstride=4,
                cstride=4,
                shade=False
            )

            papyrus_mask = masks[z] == 1
            if papyrus_mask.any():
                ax.scatter(
                    X[papyrus_mask],
                    Y[papyrus_mask],
                    Z[papyrus_mask],
                    c="red",
                    s=3,
                    alpha=alpha_mask
                )

        ax.set_xlim(0, W)
        ax.set_ylim(0, H)
        ax.set_zlim(0, D)

        ax.set_xlabel("X")
        ax.set_ylabel("Y")
        ax.set_zlabel("Slice")

        ax.view_init(elev=25, azim=azim)
        ax.set_title(f"{title_prefix} | Z {azim}°")

plt.tight_layout()
plt.show()
